In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.sandwich_covariance import cov_hac
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from arch import arch_model
from arch.univariate import ARX
from scipy.stats import norm

### Import data

In [4]:
path = os.path.abspath('E:/RA/Geert/task1.py')
dir_path = os.path.dirname(path)
os.chdir(dir_path)
excel_file = pd.ExcelFile('Aggregate_CPI_inflation_20230513.xls')
sheet_quarter = excel_file.sheet_names[0]
sheet_month = excel_file.sheet_names[1]
#quarterly and monthly aggregate CPI data (deseasonalized). The full sample is 1947-2022
data_quarter = excel_file.parse(sheet_quarter, skiprows=2)
data_month = excel_file.parse(sheet_month, skiprows=2)
data_quarter.index = pd.to_datetime(data_quarter['Year'].astype(str) + '-Q' + data_quarter['Quarter'].astype(str))
data_month.index = pd.to_datetime(data_month[['Year', 'Month']].assign(day=1))
data_quarter.columns = ['Year', 'Quarter', 'Price index', 'Inflation', 'Forecasted inflation', 'Inflation shock']
data_month.columns = ['Year', 'Month', 'Price index', 'Inflation', 'Forecasted inflation', 'Inflation shock']


C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_10068/436920217.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data_quarter.index = pd.to_datetime(data_quarter['Year'].astype(str) + '-Q' + data_quarter['Quarter'].astype(str))


In [5]:
sample_data = data_quarter[data_quarter['Year']>1969]
sample_data['Inflation_lag_1'] =  sample_data['Inflation'].shift(1)
sample_data['Inflation_lag_2'] =  sample_data['Inflation'].shift(2)
sample_data['Forecasted_inflation_lag_1'] =  sample_data['Forecasted inflation'].shift(1)
sample_data = sample_data.dropna()

C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_10068/554186344.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sample_data['Inflation_lag_1'] =  sample_data['Inflation'].shift(1)
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_10068/554186344.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sample_data['Inflation_lag_2'] =  sample_data['Inflation'].shift(2)
C:\Users\ADMINI~1\AppData\Local\Temp/ipykernel_10068/554186344.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice fro

## Model Selection

GARCH model specification:
$$
 \sigma_{t}^{2}=\omega +\sum_{i=1}^{p}\alpha_{i} \epsilon_{t-i}^2
         +\sum_{k=1}^{q}\beta_{k}\sigma_{t-k}^{2} 
$$

EGARCH model specification:
$$
 \ln\sigma_{t}^{2}=\omega +\sum_{i=1}^{p}\alpha_{i}  \left(\left|e_{t-i}\right|-\sqrt{2/\pi}\right)
        +\sum_{j=1}^{o}\gamma_{j} e_{t-j} +\sum_{k=1}^{q}\beta_{k}\ln\sigma_{t-k}^{2} 
$$
where :
$e_{t}=\epsilon_{t}/\sigma_{t}$

GJR GARCH model specification:
$$
\sigma_{t}^{2}=\omega
        + \sum_{i=1}^{p}\alpha_{i}\left|\epsilon_{t-i}\right|^{2}
        +\sum_{j=1}^{o}\gamma_{j}\left|\epsilon_{t-j}\right|^{2}
        I\left[\epsilon_{t-j}<0\right]+\sum_{k=1}^{q}\beta_{k}\sigma_{t-k}^{2}
$$


In [13]:
MeanModel = {'0,1':['Forecasted inflation'],
             '1,1':['Inflation_lag_1','Forecasted inflation'],
            '2,1':['Inflation_lag_1', 'Inflation_lag_2','Forecasted inflation'],
             '2,2':['Inflation_lag_1', 'Inflation_lag_2','Forecasted inflation','Forecasted_inflation_lag_1' ],
            }

In [72]:
def GarchFamilyResults(df):
    garch11 = arch_model(df, mean='Constant', vol='GARCH',p=1, q=1,dist='normal').fit(disp='off',cov_type='hac')
    garch21 = arch_model(df, mean='Constant', vol='GARCH',p=2, q=1,dist='normal',).fit(disp='off',cov_type='hac')
    garch12 = arch_model(df, mean='Constant', vol='GARCH',p=1, q=2,dist='normal',).fit(disp='off',cov_type='hac')
    garch22 = arch_model(df, mean='Constant', vol='GARCH',p=2, q=2,dist='normal',).fit(disp='off',cov_type='hac')
    print('\nGARCH(p,q) model')
    print('\t \t AIC: \t \t \t BIC',
          '\nGARCH(1,1): ',garch11.aic,'\t',garch11.bic,
          '\nGARCH(2,1): ',garch21.aic,'\t',garch21.bic,
          '\nGARCH(1,2): ',garch12.aic,'\t',garch12.bic,
          '\nGARCH(2,2): ',garch22.aic,'\t',garch22.bic,)

    egarch111 = arch_model(df, mean='Constant', vol='EGARCH',p=1, o=1, q=1,dist='normal',).fit(disp='off',cov_type='hac')
    egarch211 = arch_model(df, mean='Constant', vol='EGARCH',p=2, o=1, q=1,dist='normal',).fit(disp='off',cov_type='hac')
    egarch112 = arch_model(df, mean='Constant', vol='EGARCH',p=1, o=1, q=2,dist='normal',).fit(disp='off',cov_type='hac')
    egarch121 = arch_model(df, mean='Constant', vol='EGARCH',p=1, o=2, q=1,dist='normal',).fit(disp='off',cov_type='hac')
    egarch221 = arch_model(df, mean='Constant', vol='EGARCH',p=2, o=2, q=1,dist='normal',).fit(disp='off',cov_type='hac')
    egarch122 = arch_model(df, mean='Constant', vol='EGARCH',p=1, o=2, q=2,dist='normal',).fit(disp='off',cov_type='hac')
    egarch212 = arch_model(df, mean='Constant', vol='EGARCH',p=2, o=1, q=2,dist='normal',).fit(disp='off',cov_type='hac')
    egarch222 = arch_model(df, mean='Constant', vol='EGARCH',p=2, o=2, q=2,dist='normal',).fit(disp='off',cov_type='hac')
    print('\nEGARCH(p,o,q) model')
    print('\t \t AIC: \t \t \t BIC',
          '\nEGARCH(1,1,1): ',egarch111.aic,'\t',egarch111.bic,
          '\nEGARCH(2,1,1): ',egarch211.aic,'\t',egarch211.bic,
          '\nEGARCH(1,1,2): ',egarch112.aic,'\t',egarch112.bic,
          '\nEGARCH(1,2,1): ',egarch121.aic,'\t',egarch121.bic,
          '\nEGARCH(2,2,1): ',egarch221.aic,'\t',egarch221.bic,
          '\nEGARCH(2,1,2): ',egarch212.aic,'\t',egarch212.bic,
          '\nEGARCH(1,2,2): ',egarch122.aic,'\t',egarch122.bic,
          '\nEGARCH(2,2,2): ',egarch222.aic,'\t',egarch222.bic,  )
    
    gjr_garch11 = arch_model(df, mean='Constant', vol='GARCH',p=1, o=1,q=1,dist='normal').fit(disp='off',cov_type='hac')
    gjr_garch12 = arch_model(df, mean='Constant', vol='GARCH',p=1, o=1,q=2,dist='normal').fit(disp='off',cov_type='hac')
    gjr_garch21 = arch_model(df, mean='Constant', vol='GARCH',p=2, o=2,q=1,dist='normal').fit(disp='off',cov_type='hac')
    gjr_garch22 = arch_model(df, mean='Constant', vol='GARCH',p=2, o=2,q=2,dist='normal').fit(disp='off',cov_type='hac')
    print('\nGJR-GARCH(p,q) model')
    print('\t \t AIC: \t \t \t BIC',
          '\nGJR-GARCH(1,1): ',gjr_garch11.aic,'\t',gjr_garch11.bic,
          '\nGJR-GARCH(1,2): ',gjr_garch12.aic,'\t',gjr_garch12.bic,
          '\nGJR-GARCH(2,1): ',gjr_garch21.aic,'\t',gjr_garch21.bic,
          '\nGJR-GARCH(2,2): ',gjr_garch22.aic,'\t',gjr_garch22.bic,)

When estimating standard errors of Garch coefficients, this package provides three choices:
1. Classic standard errors (also known as White's or QMLE standard errors)
2. Robust standard errors (also known as Bollerslev-Wooldridge robust covariance estimator, which provides robustness to misspecification)
3. HAC (Heteroskedasticity and Autocorrelation Consistent) standard errors, suitable for models with autocorrelated errors.

I use HAC this time.

### AIC & BIC of Garch models under different mean models

I report AIC and BIC of variance models. Should I include AIC & BIC of mean model? How should I calculate this? Are below equations correct?

Calculate AIC  by 
$$
AIC_{total} = AIC_{mean model} + AIC_{variance model}
$$
Calculate BIC  by 
$$
BIC_{total} = -2 \times ( log L(mean model) + log L(variance model) ) + log(num of observations) \times (num of params)
$$

## Mean Model

$
\pi (t) =  fc(t-1) + \epsilon (t)
$

In [73]:
GarchFamilyResults(sample_data['Inflation shock'])


GARCH(p,q) model
	 	 AIC: 	 	 	 BIC 
GARCH(1,1):  389.0986444462487 	 402.448796765054 
GARCH(2,1):  373.8416214082347 	 390.52931180674125 
GARCH(1,2):  391.09864426184686 	 407.7863346603534 
GARCH(2,2):  375.8416209331775 	 395.86684941138543

EGARCH(p,o,q) model
	 	 AIC: 	 	 	 BIC 
EGARCH(1,1,1):  370.0508789531292 	 386.73856935163576 
EGARCH(2,1,1):  372.02736130090483 	 392.05258977911274 
EGARCH(1,1,2):  372.05087858002037 	 392.0761070582283 
EGARCH(1,2,1):  362.5254320444129 	 382.55066052262083 
EGARCH(2,2,1):  362.0639991869675 	 385.4267657448768 
EGARCH(2,1,2):  374.02736108457873 	 397.390127642488 
EGARCH(1,2,2):  364.52543230498486 	 387.8881988628941 
EGARCH(2,2,2):  364.0640027675429 	 390.76430740515343

GJR-GARCH(p,q) model
	 	 AIC: 	 	 	 BIC 
GJR-GARCH(1,1):  371.3048885167514 	 387.99257891525804 
GJR-GARCH(1,2):  373.3048876254667 	 393.3301161036746 
GJR-GARCH(2,1):  358.9579743134722 	 382.3207408713814 
GJR-GARCH(2,2):  360.9579742822869 	 387.65827891989744

$
\pi (t) = c +\rho \pi (t-1)   + \phi fc(t-1) + \epsilon (t)
$

In [74]:
X = sm.add_constant(sample_data[['Inflation_lag_1','Forecasted inflation']] )
y = sample_data['Inflation']
model = sm.OLS(y, X)

# HAC covariance matrix with 3 lag,  Bartlett Kernal
results = model.fit(cov_type='HAC', cov_kwds={'maxlags': 3})
GarchFamilyResults(results.resid)


GARCH(p,q) model
	 	 AIC: 	 	 	 BIC 
GARCH(1,1):  383.8561205615791 	 397.20627288038435 
GARCH(2,1):  374.2342503504929 	 390.92194074899953 
GARCH(1,2):  385.85612056475645 	 402.543810963263 
GARCH(2,2):  376.23425032132866 	 396.25947879953657

EGARCH(p,o,q) model
	 	 AIC: 	 	 	 BIC 
EGARCH(1,1,1):  367.53530843869 	 384.22299883719654 
EGARCH(2,1,1):  364.21251199024596 	 384.23774046845386 
EGARCH(1,1,2):  369.53530842271334 	 389.56053690092125 
EGARCH(1,2,1):  360.1233782674458 	 380.1486067456537 
EGARCH(2,2,1):  361.98978337032605 	 385.35254992823525 
EGARCH(2,1,2):  366.2125117446775 	 389.57527830258675 
EGARCH(1,2,2):  362.1433605271387 	 385.506127085048 
EGARCH(2,2,2):  363.99013483132154 	 390.6904394689321

GJR-GARCH(p,q) model
	 	 AIC: 	 	 	 BIC 
GJR-GARCH(1,1):  368.91907990331595 	 385.6067703018225 
GJR-GARCH(1,2):  370.91907991659843 	 390.94430839480634 
GJR-GARCH(2,1):  361.1641537210196 	 384.52692027892886 
GJR-GARCH(2,2):  363.16415380928186 	 389.864458446

$
\pi (t) = c +\rho_1 \pi (t-1)  +\rho_2 \pi (t-2)  + \phi fc(t-1) + \epsilon (t)
$

In [75]:
X = sm.add_constant(sample_data[['Inflation_lag_1', 'Inflation_lag_2','Forecasted inflation']] )
y = sample_data['Inflation']
model = sm.OLS(y, X)

# HAC covariance matrix with 3 lag,  Bartlett Kernal
results = model.fit(cov_type='HAC', cov_kwds={'maxlags': 3})
GarchFamilyResults(results.resid)


GARCH(p,q) model
	 	 AIC: 	 	 	 BIC 
GARCH(1,1):  382.3388167217096 	 395.68896904051485 
GARCH(2,1):  373.16938271989085 	 389.85707311839747 
GARCH(1,2):  384.33881663447625 	 401.02650703298286 
GARCH(2,2):  375.1693826118289 	 395.1946110900368

EGARCH(p,o,q) model
	 	 AIC: 	 	 	 BIC 
EGARCH(1,1,1):  368.0935233777408 	 384.78121377624734 
EGARCH(2,1,1):  363.9524969026072 	 383.9777253808151 
EGARCH(1,1,2):  370.09352337130474 	 390.11875184951265 
EGARCH(1,2,1):  359.76061484656475 	 379.78584332477266 
EGARCH(2,2,1):  361.719810242215 	 385.08257680012423 
EGARCH(2,1,2):  365.9524967673258 	 389.31526332523504 
EGARCH(1,2,2):  361.76061497114233 	 385.1233815290516 
EGARCH(2,2,2):  363.7198094806834 	 390.42011411829395

GJR-GARCH(p,q) model
	 	 AIC: 	 	 	 BIC 
GJR-GARCH(1,1):  370.1223717999684 	 386.81006219847495 
GJR-GARCH(1,2):  372.12237179899284 	 392.14760027720075 
GJR-GARCH(2,1):  360.1302230213756 	 383.4929895792849 
GJR-GARCH(2,2):  362.13022265390634 	 388.8305272

$
\pi (t) = c +\rho_1 \pi (t-1)  +\rho_2 \pi (t-2)  + \phi_1 fc(t-1) + \phi_2 fc(t-2) + \epsilon (t)
$

In [76]:
X = sm.add_constant(sample_data[['Inflation_lag_1', 'Inflation_lag_2','Forecasted inflation','Forecasted_inflation_lag_1' ]] )
y = sample_data['Inflation']
model = sm.OLS(y, X)

# HAC covariance matrix with 3 lag,  Bartlett Kernal
results = model.fit(cov_type='HAC', cov_kwds={'maxlags': 3})
GarchFamilyResults(results.resid)


GARCH(p,q) model
	 	 AIC: 	 	 	 BIC 
GARCH(1,1):  382.63818754417355 	 395.9883398629788 
GARCH(2,1):  373.7368631853851 	 390.4245535838917 
GARCH(1,2):  384.6381875321643 	 401.3258779306709 
GARCH(2,2):  375.736863786458 	 395.7620922646659

EGARCH(p,o,q) model
	 	 AIC: 	 	 	 BIC 
EGARCH(1,1,1):  367.50328258163023 	 384.19097298013685 
EGARCH(2,1,1):  363.1883514496542 	 383.2135799278621 
EGARCH(1,1,2):  369.50328309533995 	 389.52851157354786 
EGARCH(1,2,1):  359.6850798442668 	 379.7103083224747 
EGARCH(2,2,1):  361.6012393210231 	 384.9640058789323 
EGARCH(2,1,2):  365.18835151984115 	 388.5511180777504 
EGARCH(1,2,2):  361.68848347798917 	 385.0512500358984 
EGARCH(2,2,2):  363.6012394544333 	 390.30154409204386

GJR-GARCH(p,q) model
	 	 AIC: 	 	 	 BIC 
GJR-GARCH(1,1):  370.659690986361 	 387.3473813848676 
GJR-GARCH(1,2):  372.6596907111827 	 392.6849191893906 
GJR-GARCH(2,1):  360.6377287956521 	 384.0004953535613 
GJR-GARCH(2,2):  362.6377292885147 	 389.33803392612526


### Present MLE details of best garch model in each mean models

$
\pi (t) =  fc(t-1) + \epsilon (t)
$

In [6]:
am = arch_model(sample_data['Inflation shock'], mean='Constant', vol='GARCH',p=2, o=2,q=1,dist='t').fit(disp='off',cov_type='hac')
am.summary()

D:\anaconda\lib\site-packages\arch\univariate\base.py:1891: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if isinstance(val[pos], np.float64):
D:\anaconda\lib\site-packages\arch\univariate\base.py:1892: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  converted = format_float_fixed(val[pos], *formats[i])


<class 'statsmodels.iolib.summary.Summary'>
"""
                      Constant Mean - GJR-GARCH Model Results                       
====================================================================================
Dep. Variable:              Inflation shock   R-squared:                       0.000
Mean Model:                   Constant Mean   Adj. R-squared:                  0.000
Vol Model:                        GJR-GARCH   Log-Likelihood:               -166.581
Distribution:      Standardized Student's t   AIC:                           349.162
Method:                  Maximum Likelihood   BIC:                           375.939
                                              No. Observations:                  210
Date:                      Wed, Nov 08 2023   Df Residuals:                      209
Time:                              10:35:13   Df Model:                            1
                                Mean Model                                
==========================================================================
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu             0.0674  3.153e-02      2.137  3.256e-02 [5.596e-03,  0.129]
                              Volatility Model                             
===========================================================================
                 coef    std err          t      P>|t|     95.0% Conf. Int.
---------------------------------------------------------------------------
omega          0.1196  3.586e-02      3.334  8.572e-04  [4.927e-02,  0.190]
alpha[1]       0.4237      0.231      1.835  6.654e-02 [-2.891e-02,  0.876]
alpha[2]       0.8358      0.345      2.424  1.533e-02    [  0.160,  1.511]
gamma[1]      -0.2006      0.258     -0.777      0.437    [ -0.707,  0.305]
gamma[2]      -0.8317      0.332     -2.503  1.231e-02    [ -1.483, -0.181]
beta[1]        0.0738      0.122      0.606      0.544    [ -0.165,  0.312]
                              Distribution                              
========================================================================
                 coef    std err          t      P>|t|  95.0% Conf. Int.
------------------------------------------------------------------------
nu             5.5788      2.019      2.764  5.713e-03 [  1.623,  9.535]
========================================================================

Covariance estimator: hac
"""

$
\pi (t) = c +\rho \pi (t-1)   + \phi fc(t-1) + \epsilon (t)
$

In [79]:
X = sm.add_constant(sample_data[['Inflation_lag_1','Forecasted inflation']] )
y = sample_data['Inflation']
model = sm.OLS(y, X)

# HAC covariance matrix with 3 lag,  Bartlett Kernal
results = model.fit(cov_type='HAC', cov_kwds={'maxlags': 3})
am = arch_model(results.resid, mean='Constant', vol='EGARCH',p=1, o=2, q=1,dist='normal',).fit(disp='off',cov_type='hac')
am.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                     Constant Mean - EGARCH Model Results                     
==============================================================================
Dep. Variable:                   None   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                     EGARCH   Log-Likelihood:               -174.062
Distribution:                  Normal   AIC:                           360.123
Method:            Maximum Likelihood   BIC:                           380.149
                                        No. Observations:                  208
Date:                Wed, May 24 2023   Df Residuals:                      207
Time:                        17:03:20   Df Model:                            1
                                Mean Model                                
==========================================================================
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu            -0.0444  3.446e-02     -1.287      0.198 [ -0.112,2.318e-02]
                              Volatility Model                             
===========================================================================
                 coef    std err          t      P>|t|     95.0% Conf. Int.
---------------------------------------------------------------------------
omega         -0.3373      0.126     -2.675  7.482e-03 [ -0.584,-9.011e-02]
alpha[1]       0.4761      0.140      3.399  6.761e-04    [  0.202,  0.751]
gamma[1]       0.0584      0.115      0.510      0.610    [ -0.166,  0.283]
gamma[2]       0.3396      0.100      3.392  6.931e-04    [  0.143,  0.536]
beta[1]        0.6995      0.101      6.913  4.749e-12    [  0.501,  0.898]
===========================================================================

Covariance estimator: hac
"""

$
\pi (t) = c +\rho_1 \pi (t-1)  +\rho_2 \pi (t-2)  + \phi fc(t-1) + \epsilon (t)
$

In [80]:
X = sm.add_constant(sample_data[['Inflation_lag_1', 'Inflation_lag_2','Forecasted inflation']] )
y = sample_data['Inflation']
model = sm.OLS(y, X)

# HAC covariance matrix with 3 lag,  Bartlett Kernal
results = model.fit(cov_type='HAC', cov_kwds={'maxlags': 3})
am = arch_model(results.resid, mean='Constant', vol='EGARCH',p=1, o=2, q=1,dist='normal',).fit(disp='off',cov_type='hac')
am.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                     Constant Mean - EGARCH Model Results                     
==============================================================================
Dep. Variable:                   None   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                     EGARCH   Log-Likelihood:               -173.880
Distribution:                  Normal   AIC:                           359.761
Method:            Maximum Likelihood   BIC:                           379.786
                                        No. Observations:                  208
Date:                Wed, May 24 2023   Df Residuals:                      207
Time:                        17:04:59   Df Model:                            1
                                Mean Model                                
==========================================================================
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu            -0.0480  3.446e-02     -1.392      0.164 [ -0.116,1.956e-02]
                              Volatility Model                             
===========================================================================
                 coef    std err          t      P>|t|     95.0% Conf. Int.
---------------------------------------------------------------------------
omega         -0.3073      0.116     -2.640  8.295e-03 [ -0.535,-7.913e-02]
alpha[1]       0.5248      0.158      3.326  8.802e-04    [  0.216,  0.834]
gamma[1]       0.0102      0.118  8.648e-02      0.931    [ -0.222,  0.242]
gamma[2]       0.3434  9.727e-02      3.530  4.150e-04    [  0.153,  0.534]
beta[1]        0.7244  9.318e-02      7.774  7.621e-15    [  0.542,  0.907]
===========================================================================

Covariance estimator: hac
"""

$
\pi (t) = c +\rho_1 \pi (t-1)  +\rho_2 \pi (t-2)  + \phi_1 fc(t-1) + \phi_2 fc(t-2) + \epsilon (t)
$

In [81]:
X = sm.add_constant(sample_data[['Inflation_lag_1', 'Inflation_lag_2','Forecasted inflation','Forecasted_inflation_lag_1' ]] )
y = sample_data['Inflation']
model = sm.OLS(y, X)

# HAC covariance matrix with 3 lag,  Bartlett Kernal
results = model.fit(cov_type='HAC', cov_kwds={'maxlags': 3})
am = arch_model(results.resid, mean='Constant', vol='EGARCH',p=1, o=2, q=1,dist='normal',).fit(disp='off',cov_type='hac')
am.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                     Constant Mean - EGARCH Model Results                     
==============================================================================
Dep. Variable:                   None   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                     EGARCH   Log-Likelihood:               -173.843
Distribution:                  Normal   AIC:                           359.685
Method:            Maximum Likelihood   BIC:                           379.710
                                        No. Observations:                  208
Date:                Wed, May 24 2023   Df Residuals:                      207
Time:                        17:06:35   Df Model:                            1
                                Mean Model                                
==========================================================================
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu            -0.0479  3.484e-02     -1.376      0.169 [ -0.116,2.035e-02]
                              Volatility Model                             
===========================================================================
                 coef    std err          t      P>|t|     95.0% Conf. Int.
---------------------------------------------------------------------------
omega         -0.3051      0.119     -2.560  1.046e-02 [ -0.539,-7.153e-02]
alpha[1]       0.5150      0.157      3.273  1.063e-03    [  0.207,  0.823]
gamma[1]   7.5207e-03      0.118  6.382e-02      0.949    [ -0.223,  0.238]
gamma[2]       0.3446  9.672e-02      3.563  3.670e-04    [  0.155,  0.534]
beta[1]        0.7265  9.556e-02      7.603  2.903e-14    [  0.539,  0.914]
===========================================================================

Covariance estimator: hac
"""